# Q-Learning Othello

## Experiments and Written Questions


### Question 1

Derive the "opponent to move" target from the zero-sum property. Why is it max and not min?

**Answer:**

The game is zero-sum, so however good a position is for the opponent, it is equally bad for us: V_me(s') = -V_opp(s'). The opponent picks their best move, so V_opp(s') = max Q(s', a'), which gives target = r - γ max Q(s', a'). 

We use max, because these scores are recorded from the opponent's point of view, and the opponent chooses the highest score for themselves.


### Question 2

Train 6x6 tabular. Report the number of states stored and the win rate vs random. Why does 6x6 fail when 4x4 succeeds? Roughly what fraction of positions in an evaluation game has the agent ever seen?

**Answer:**

The 6x6 agent stored 712,265 states but won only 46.4% of games against random, so it was no better than random play. 4x4 works because the board is small and the agent sees 99.9% of positions during evaluation. On 6x6, it has seen only 27.6% of them. A table cannot generalize, so the agent has to guess on every new position. 

### Question 3

Train 4x4 with no exploration (`--eps-start 0 --eps-end 0`), then with pure random play (`--eps-start 1 --eps-end 1`). Report both results. Explain each, using the fact that Q-learning is off-policy.

**Answer:**

With no exploration, the agent saw only 2,395 states and won 78.5% of games against random, because it kept repeating moves it already liked. 

With pure random play, it saw 8,644 states and won 97.5%. This works because Q-learning is off-policy. The agent can move randomly while still learning the value of the best move, because its target uses the max instead of the move it actually plays next. 



### Question 4

Print the 8x8 mid-game weights (the `w_mid` line during training) next to the squares each class covers. Which classes are positive and which negative? Relate this to standard Othello strategy (corners, X-squares, edges). To list the classes:


In [2]:
import json
import othello_q as M

g = M.Othello(8)

# Load the mid-game weights from the trained model.
with open('q_linear_8.json') as file:
    w_mid = json.load(file)['w'][1]

names = ['Corner', 'C-square', 'Edge', 'Edge', 'X-square',
         'Inner', 'Inner', 'Inner', 'Inner', 'Center']

# Print each square class next to its learned weight.
for k, squares in enumerate(g.classes):
    labels = ['abcdefgh'[i % 8] + str(i // 8 + 1) for i in squares]
    print(f'{k}  {names[k]:<9} {w_mid[k]:+.2f}   {", ".join(labels)}')



0  Corner    +1.71   a1, h1, a8, h8
1  C-square  -0.32   b1, g1, a2, h2, a7, h7, b8, g8
2  Edge      +0.42   c1, f1, a3, h3, a6, h6, c8, f8
3  Edge      +0.11   d1, e1, a4, h4, a5, h5, d8, e8
4  X-square  -0.32   b2, g2, b7, g7
5  Inner     -0.06   c2, f2, b3, g3, b6, g6, c7, f7
6  Inner     +0.07   d2, e2, b4, g4, b5, g5, d7, e7
7  Inner     +0.06   c3, f3, c6, f6
8  Inner     +0.03   d3, e3, c4, f4, c5, f5, d6, e6
9  Center    -0.08   d4, e4, d5, e5


**Answer:**

Corners have the highest weight (+1.71) because a corner disc can never be flipped. The other edge squares are also positive (+0.42, +0.11) because they are hard to flip. X-squares and C-squares are negative (-0.32) because they sit next to a corner and can give the opponent that corner. The inner squares are close to zero, and this pattern matches standard Othello strategy.

### Question 5

Train 8x8 with `--alpha 0.1` and `--alpha 0.001`. Compare the learning curves to the default of 0.01, and explain the trade-off.

**Answer:**

Alpha 0.1 learns fastest, reaching 92.5% against greedy after only 600 games, but it is unstable and swings between 80% and 95.5%. 

Alpha 0.001 is too slow and is still at only 64% after 6,000 games. 

Alpha 0.01 is the best balance, improving from 61.5% to 97%. The trade-off is that a large alpha learns quickly but overreacts to each game, while a small alpha is steadier but needs many more games.


### Question 6

Remove the opponent-mobility feature from `LinearQ.features` and retrain. How much does performance against greedy change? Why might mobility matter more than disc count in Othello?

**Answer:**

Removing the opponent-mobility feature lowered the win rate against greedy from 95.2% to 92.4%, a drop of 2.8 percentage points over 500 games. This gap is small and could be training noise, since another full-feature run with the same settings scored 91.4%. 

Mobility can still matter more than disc count because discs can flip many times, so being ahead in the middle of the game does not guarantee a win. Limiting the opponent's moves can force them into bad moves, such as giving up a corner.